# PCB Vision AI — Google Colab GPU Model Training

This notebook trains the deep learning model (YOLOv8/YOLO11) for **Intelligent PCB Defect Detection and Classification** on a free Google Colab GPU (T4/V100/A100).

### Six Target Defect Classes (DeepPCB Benchmark):
1. **Open circuit** (`open`)
2. **Short circuit** (`short`)
3. **Mouse bite** (`mousebite`)
4. **Spur** (`spur`)
5. **Spurious copper** (`copper`)
6. **Pin hole** (`pin-hole`)


## 1. Verify GPU Acceleration
Ensure **Runtime > Change runtime type** is set to **T4 GPU** or higher.

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected! Please enable GPU runtime under Runtime > Change runtime type.')

## 2. Install Required Dependencies

In [ ]:
!pip install --upgrade ultralytics opencv-python pyyaml

## 3. Clone DeepPCB Dataset and Setup Workspace

In [ ]:
import os, shutil
from pathlib import Path

if not Path('DeepPCB').exists():
    !git clone https://github.com/tangsanli5201/DeepPCB.git
print('DeepPCB repository ready.')

## 4. Convert DeepPCB to YOLO Format with Group Splitting
Splits the 11 PCB board groups into train (7 groups, 1070 images), val (2 groups, 253 images), and test (2 groups, 177 images) without spatial data leakage.

In [ ]:
import cv2, yaml, random
from collections import defaultdict

CLASSES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
raw = Path('DeepPCB/PCBData')
out = Path('dataset')
seed = 42

candidates = []
for img in raw.rglob('*_test.jpg'):
    label = img.parent.parent / (img.parent.name + '_not') / img.name.replace('_test.jpg', '.txt')
    if label.exists():
        candidates.append((img, label))

groups = defaultdict(list)
for im, lab in candidates:
    rel = im.relative_to(raw)
    g = rel.parts[0] if len(rel.parts) > 1 else im.stem
    groups[g].append((im, lab))

keys = sorted(groups)
rng = random.Random(seed)
rng.shuffle(keys)

n = len(keys)
ntest = max(1, round(n * 0.15))
nval = max(1, round(n * 0.15))
split_keys = {'test': keys[:ntest], 'val': keys[ntest:ntest+nval], 'train': keys[ntest+nval:]}

for split, selected in split_keys.items():
    im_dir = out / 'images' / split
    lb_dir = out / 'labels' / split
    im_dir.mkdir(parents=True, exist_ok=True)
    lb_dir.mkdir(parents=True, exist_ok=True)
    for g in selected:
        for im, lab in groups[g]:
            arr = cv2.imread(str(im))
            h, w = arr.shape[:2]
            converted = []
            for line in lab.read_text(encoding='utf-8-sig').splitlines():
                if not line.strip(): continue
                parts = line.replace(',', ' ').split()
                x1, y1, x2, y2, cls = (int(float(a)) for a in parts)
                cx = (x1 + x2) / (2 * w)
                cy = (y1 + y2) / (2 * h)
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h
                converted.append(f'{cls - 1} {cx:.8f} {cy:.8f} {bw:.8f} {bh:.8f}')
            dest = f'{g}_{im.stem}'
            shutil.copy2(im, im_dir / f'{dest}.jpg')
            (lb_dir / f'{dest}.txt').write_text('\n'.join(converted) + ('\n' if converted else ''))

config = {
    'path': str(out.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': dict(enumerate(CLASSES))
}
(out / 'data.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
print('Dataset prepared successfully. Total groups:', len(keys))
print('Split distribution:', {k: len(v) for k, v in split_keys.items()})

## 5. Train YOLOv8 Model on GPU
We train `yolov8n.pt` for 50 epochs with batch size 16 at resolution 640x640 with early stopping.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    patience=12,
    project='runs/pcb',
    name='train',
    plots=True
)
print('Training complete! Best model saved at runs/pcb/train/weights/best.pt')

## 6. Evaluate on Held-Out Test Split (177 Images)
Measure mAP@50, mAP@50-95, per-class Precision, Recall, and confusion matrix.

In [ ]:
best_model = YOLO('runs/pcb/train/weights/best.pt')
metrics = best_model.val(data='dataset/data.yaml', split='test', plots=True, project='runs/pcb', name='test')
print('=== TEST SET BENCHMARK RESULTS ===')
print(f'Test mAP@50     : {metrics.box.map50:.4f}')
print(f'Test mAP@50-95  : {metrics.box.map:.4f}')
print(f'Mean Precision  : {metrics.box.mp:.4f}')
print(f'Mean Recall     : {metrics.box.mr:.4f}')

## 7. Package and Download Weights
Download `best.pt` to copy to your local machine at `runs/pcb/train/weights/best.pt`.

In [ ]:
from google.colab import files
import zipfile

# Direct best.pt download
files.download('runs/pcb/train/weights/best.pt')

# Archive complete training run with plots and metrics
with zipfile.ZipFile('pcb_yolo_run.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in Path('runs/pcb').rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to('runs/pcb'))
files.download('pcb_yolo_run.zip')
print('Downloads triggered.')